# PancScan AI — Data Exploration (Step 2)

This notebook loads a single case from the **Medical Segmentation Decathlon Task07_Pancreas** dataset, prints basic metadata, and saves axial-slice visualizations with pancreas (blue) and tumor (red) overlays.

**Prerequisite:** Extract `Task07_Pancreas.tar` into `./data/Task07_Pancreas/`.

In [ ]:
import sys
from pathlib import Path

# Add project root so `import src...` works from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import (
    get_default_data_dir,
    load_dataset_manifest,
    load_scan_with_label,
    print_scan_info,
    resolve_training_paths,
    visualize_axial_slices,
)

DATA_DIR = get_default_data_dir()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")

## 1. Inspect dataset manifest

In [ ]:
manifest = load_dataset_manifest(DATA_DIR)
print(f"Dataset name   : {manifest['name']}")
print(f"Description    : {manifest['description']}")
print(f"License        : {manifest['licence']}")
print(f"Training cases : {manifest['numTraining']}")
print(f"Test cases     : {manifest['numTest']}")
print(f"Labels         : {manifest['labels']}")

## 2. Load one scan + label map

In [ ]:
# Change CASE_INDEX to explore different patients (0 .. numTraining-1)
CASE_INDEX = 0

entry = manifest["training"][CASE_INDEX]
image_path = resolve_training_paths(DATA_DIR, entry["image"])
label_path = resolve_training_paths(DATA_DIR, entry["label"])

scan = load_scan_with_label(image_path, label_path)
print_scan_info(scan)

## 3. Quick numeric sanity checks

In [ ]:
import numpy as np

img = scan.image.data
lbl = scan.label.data

print("Image dtype/shape:", img.dtype, img.shape)
print("Label dtype/shape:", lbl.dtype, lbl.shape)
print("Unique label values:", np.unique(lbl))
print("Voxel spacing (mm):", scan.image.voxel_spacing)

# Fraction of voxels per class
total = lbl.size
for class_id, name in [(0, "background"), (1, "pancreas"), (2, "tumor")]:
    frac = 100.0 * np.sum(lbl == class_id) / total
    print(f"  {name:12s}: {frac:6.3f}% of voxels")

## 4. Visualize axial slices

Top row: windowed CT (-100 to 240 HU).  
Bottom row: overlay — **blue = pancreas**, **red = tumor**.

In [ ]:
from IPython.display import Image, display

out_path = OUTPUT_DIR / f"{scan.patient_id}_axial_slices.png"
visualize_axial_slices(scan, out_path, num_slices=4)
print(f"Saved: {out_path}")
display(Image(filename=str(out_path)))

## 5. (Optional) Load a case likely to contain tumor

Not every training volume has tumor annotations. The loop below finds the first case with tumor voxels and visualizes it.

In [ ]:
from src.data_loading import LABEL_TUMOR

tumor_case = None
for i, item in enumerate(manifest["training"][:50]):  # scan first 50 for speed
    ip = resolve_training_paths(DATA_DIR, item["image"])
    lp = resolve_training_paths(DATA_DIR, item["label"])
    candidate = load_scan_with_label(ip, lp)
    if (candidate.label.data == LABEL_TUMOR).any():
        tumor_case = candidate
        print(f"Found tumor in case index {i}: {candidate.patient_id}")
        break

if tumor_case is not None:
    print_scan_info(tumor_case)
    tumor_out = OUTPUT_DIR / f"{tumor_case.patient_id}_tumor_axial_slices.png"
    visualize_axial_slices(tumor_case, tumor_out, num_slices=4)
    display(Image(filename=str(tumor_out)))
else:
    print("No tumor found in first 50 cases (unexpected — widen search if needed).")